# 03 — Derivatives: Advanced Models

This notebook compares AbaQuant's advanced option-pricing model family:
Black–Scholes, Cox–Ross–Rubinstein (CRR) trees, Bachelier (normal
volatility), Heston (stochastic volatility), Merton jump-diffusion,
Normal-Inverse-Gaussian (NIG), SABR, and Variance-Gamma. It also covers
model diagnostics, Monte Carlo/path simulation, and figures.

> **Assumption reminder:** each advanced model relaxes a different
> Black–Scholes assumption (jumps, stochastic vol, heavy tails, ...). Always
> check a model's parameter constraints and limiting cases before comparing
> outputs across models — see `docs/domains/assumptions.rst`.

**Sections:**
1. Build one instance of each pricing model
2. Price all models
3. Model diagnostics
4. Simulations (Monte Carlo, GBM, Merton, Lévy)
5. Visualizations


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
import numpy as np

from abaquant.derivatives.analytics import distributions, parity, volatility
from abaquant.derivatives.models import (
    BlackScholesMertonModel,
    CoxRossRubinsteinModel,
    HestonStochasticVolatilityModel,
    MertonJumpDiffusionModel,
    NormalBachelierModel,
    NormalInverseGaussianModel,
    SABRVolatilityModel,
    VarianceGammaProcessModel,
)
from abaquant.derivatives.models.binomial import crr_tree_parameters
from abaquant.derivatives.models.black_scholes import bsm_d1_d2_summary
from abaquant.derivatives.models.merton import merton_jump_statistics
from abaquant.derivatives.monte_carlo import monte_carlo_bsm
from abaquant.derivatives.numerics.implied_volatility import implied_volatility_black_scholes
from abaquant.derivatives.simulation.gbm import simulate_gbm_paths
from abaquant.derivatives.simulation.levy import simulate_vg_nig_returns
from abaquant.derivatives.simulation.merton import simulate_merton_paths
from abaquant.visualization import VisualizationError

## 1. Build one instance of each pricing model

All models below share the same underlying spot (100), strike (100), and
1-year maturity so their outputs are directly comparable.


In [ ]:
models = {
    "black_scholes": BlackScholesMertonModel(100.0, 100.0, 1.0, 0.05, 0.20),
    "crr": CoxRossRubinsteinModel(100.0, 100.0, 1.0, 0.05, 0.20, number_of_steps=50),
    "bachelier": NormalBachelierModel(100.0, 100.0, 1.0, 0.05, 20.0),
    "heston": HestonStochasticVolatilityModel(
        100.0, 100.0, 1.0, 0.05, 0.0, 0.04, 2.0, 0.04, 0.3, -0.5
    ),
    "merton": MertonJumpDiffusionModel(100.0, 100.0, 1.0, 0.05, 0.20, poisson_series_terms=8),
    "nig": NormalInverseGaussianModel(100.0, 100.0, 1.0, 0.05, 5.0, 0.0, 0.2),
    "sabr": SABRVolatilityModel(100.0, 100.0, 1.0, 0.20, 0.5, -0.3, 0.4),
    "variance_gamma": VarianceGammaProcessModel(100.0, 100.0, 1.0, 0.05, 0.20, -0.1, 0.2),
}
list(models.keys())

## 2. Price all models

Call prices (or, for SABR, the ATM implied volatility) across the model family.


In [ ]:
prices_by_model = {
    "black_scholes_call": models["black_scholes"].call_price(),
    "crr_put": models["crr"].put_price(),
    "bachelier_call": models["bachelier"].call_price(),
    "heston_call": models["heston"].call_price(),
    "merton_call": models["merton"].call_price(),
    "nig_call": models["nig"].call_price(),
    "sabr_implied_volatility": models["sabr"].implied_vol(),
    "variance_gamma_call": models["variance_gamma"].call_price(),
}
for name, value in prices_by_model.items():
    print(f"{name:26s}: {value:.6f}")

## 3. Model diagnostics

Analytical d1/d2 statistics, full scalar diagnostics reports, CRR tree
parameters, Merton jump statistics, sample-distribution moments, a
put-call-parity check, realized volatility, and a numerically solved
implied volatility.


In [ ]:
bsm_price = models["black_scholes"].call_price()
prices = np.array([100.0, 101.0, 99.0, 102.0, 103.0])

diagnostics = {
    "bsm_d1_d2": bsm_d1_d2_summary(100.0, 100.0, 1.0, 0.05, 0.20),
    "crr_tree_parameters": crr_tree_parameters(1.0, 0.05, 0.20, N=50),
    "merton_jump_statistics": merton_jump_statistics(1.0, -0.05, 0.20, 0.20),
    "distribution_moments": distributions.distribution_moments(prices),
    "parity_check": parity.parity_check(10.0, 7.0, 100.0, 100.0, 1.0, 0.05),
    "realized_volatility_last": float(volatility.realized_vol(prices, window=2)[-1]),
    "solved_bsm_iv": implied_volatility_black_scholes(bsm_price, 100.0, 100.0, 1.0, 0.05),
}
for key, value in diagnostics.items():
    print(f"{key}: {value}")

In [ ]:
call_diagnostics = models["black_scholes"].diagnostics("call").as_dict()
put_diagnostics = models["black_scholes"].diagnostics("put").as_dict()
call_diagnostics

## 4. Simulations

Monte Carlo option pricing under Black–Scholes, plus raw path simulation
under GBM, Merton jump-diffusion, and the Lévy-family (Variance-Gamma / NIG)
return simulators.


In [ ]:
simulations = {
    "monte_carlo_bsm": monte_carlo_bsm(100.0, 100.0, 1.0, 0.05, 0.20, n_paths=2_000),
    "gbm_shape": simulate_gbm_paths(100.0, 1.0, 0.05, 0.20, n_paths=8, n_steps=20)["paths"].shape,
    "merton_shape": simulate_merton_paths(100.0, 1.0, 0.05, 0.20, n_paths=8, n_steps=20)["paths"].shape,
    "levy_keys": sorted(
        simulate_vg_nig_returns(1.0, 0.20, -0.1, 0.2, 5.0, 0.0, 0.2, 0.2, n_sim=500).keys()
    ),
}
for key, value in simulations.items():
    print(f"{key}: {value}")

## 5. Visualizations

Payoff curves, price profiles, extrinsic value, standardized Greeks, price
and delta surfaces, a CRR lattice, and the SABR volatility smile.


In [ ]:
try:
    figures = {
        "bsm_call_payoff": models["black_scholes"].visualize(
            chart="payoff", option_type="call"
        ),
        "bsm_put_profile": models["black_scholes"].visualize(
            chart="price_profile", option_type="put"
        ),
        "bsm_call_extrinsic": models["black_scholes"].visualize(
            chart="extrinsic_value", option_type="call"
        ),
        "bsm_call_greeks": models["black_scholes"].visualize(
            chart="greeks", option_type="call", greek_scale="standardized"
        ),
        "crr_lattice": CoxRossRubinsteinModel(
            100.0, 100.0, 1.0, 0.05, 0.20, number_of_steps=6
        ).visualize(chart="tree", option_type="put"),
        "sabr_smile": models["sabr"].visualize(chart="volatility_smile"),
    }
    print(f"Created {len(figures)} figures: {list(figures)}")
except VisualizationError as exc:
    print(f"Visualization skipped (optional dependency missing): {exc}")

## Takeaway

Each advanced model trades off tractability for extra realism (jumps,
stochastic vol, heavy tails, negative rates). Use `abaquant.derivatives.comparison.compare_all_models`
when you want a single call to price the same contract under several models
side by side, and always sanity-check calibrated parameters — see the
**Assumptions and limitations** guide in the docs.
